In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [4]:
%%writefile part1_parallel_stack.cu
// part1_parallel_stack.cu — Практическая работа №5: Параллельный стек (LIFO) на CUDA
#include <iostream>      // Для вывода результатов и отладки (Лекция №1: базовый ввод-вывод)
#include <random>        // Для генерации случайных чисел (mt19937 и uniform_int_distribution) — Лекция №1: важность случайных данных для тестов
#include <chrono>        // Для замера времени выполнения (Лекция №4: анализ производительности)
#include <cuda_runtime.h> // CUDA API: память, ядра, атомарные операции (Лекция №3: основы CUDA, Лекция №5: параллельные структуры)

using namespace std;     // Упрощает код (Лекция №1: стандартная практика)

#define CUDA_CHECK(err) do { \
    cudaError_t local_err = (err); \
    if (local_err != cudaSuccess) { \
        cerr << "CUDA error: " << cudaGetErrorString(local_err) << " at line " << __LINE__ << endl; \
        exit(1); \
    } \
} while(0)

// Константы
const int MAX_STACK_SIZE = 1000000;  // Максимальный размер стека (Лекция №5: фиксированный размер для GPU)
const int BLOCK_SIZE = 256;          // Размер блока (Лекция №3: оптимальный размер для occupancy)

// Глобальная память для стека и указателя вершины (Лекция №5: глобальная память для общей структуры)
__device__ int stack[MAX_STACK_SIZE];
__device__ int top = -1;  // Указатель вершины — атомарно обновляется (Лекция №5: атомарный доступ)

// Ядро для параллельного push в стек (Лекция №5: безопасное добавление элементов)
__global__ void parallel_push(int *values, int num_pushes) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;  // Глобальный индекс потока (Лекция №3: индексация)
    if (idx >= num_pushes) return;  // Проверка границ (Лекция №3: безопасность потоков)

    int val = values[idx];  // Значение для push (Лекция №5: данные для стека)

    // Атомарное увеличение вершины и запись значения (Лекция №5: atomicAdd для безопасного push)
    int pos = atomicAdd(&top, 1);  // Атомарно увеличиваем top (Лекция №5: атомарные операции)
    if (pos < MAX_STACK_SIZE) {    // Проверка переполнения (Лекция №5: защита от переполнения)
        stack[pos] = val;          // Запись значения в стек (Лекция №4: глобальная память)
    }
}

// Ядро для параллельного pop из стека (Лекция №5: безопасное извлечение элементов)
__global__ void parallel_pop(int *results, int num_pops) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;  // Глобальный индекс
    if (idx >= num_pops) return;  // Проверка границ

    int pos = atomicSub(&top, 1);  // Атомарно уменьшаем top (Лекция №5: atomicSub для pop)
    if (pos >= 0) {                // Проверка на пустоту
        results[idx] = stack[pos]; // Извлечение значения (Лекция №5: атомарный pop)
    } else {
        results[idx] = -1;         // Ошибка — стек пуст (Лекция №5: обработка underflow)
    }
}

int main() {
    cout << "Parallel Stack (LIFO) on GPU\n";  // Заголовок (Лекция №5: демонстрация параллельного стека)

    const int NUM_OPERATIONS = 1000000;  // Количество операций push/pop (Лекция №5: большое количество для теста)
    int *h_values, *d_values;            // Массивы для значений
    int *h_results, *d_results;          // Результаты pop

    // Выделение на CPU
    h_values = new int[NUM_OPERATIONS];
    h_results = new int[NUM_OPERATIONS];

    // Заполнение случайными значениями (Лекция №1: случайные данные для тестов)
    mt19937 gen(time(nullptr));
    uniform_int_distribution<int> dist(1, 1000000);
    for (int i = 0; i < NUM_OPERATIONS; ++i) h_values[i] = dist(gen);

    // Выделение на GPU
    CUDA_CHECK(cudaMalloc(&d_values, NUM_OPERATIONS * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_results, NUM_OPERATIONS * sizeof(int)));

    // Копирование значений на GPU
    CUDA_CHECK(cudaMemcpy(d_values, h_values, NUM_OPERATIONS * sizeof(int), cudaMemcpyHostToDevice));

    dim3 threads(BLOCK_SIZE);
    dim3 blocks((NUM_OPERATIONS + BLOCK_SIZE - 1) / BLOCK_SIZE);

    // Замер времени push
    auto start_push = chrono::high_resolution_clock::now();
    parallel_push<<<blocks, threads>>>(d_values, NUM_OPERATIONS);
    CUDA_CHECK(cudaDeviceSynchronize());
    auto end_push = chrono::high_resolution_clock::now();
    chrono::duration<double> push_time = end_push - start_push;

    // Замер времени pop
    auto start_pop = chrono::high_resolution_clock::now();
    parallel_pop<<<blocks, threads>>>(d_results, NUM_OPERATIONS);
    CUDA_CHECK(cudaDeviceSynchronize());
    auto end_pop = chrono::high_resolution_clock::now();
    chrono::duration<double> pop_time = end_pop - start_pop;

    // Копирование результатов pop обратно
    CUDA_CHECK(cudaMemcpy(h_results, d_results, NUM_OPERATIONS * sizeof(int), cudaMemcpyDeviceToHost));

    // Освобождение памяти
    delete[] h_values;
    delete[] h_results;
    CUDA_CHECK(cudaFree(d_values));
    CUDA_CHECK(cudaFree(d_results));

    cout << "Parallel Push time: " << push_time.count() << " seconds\n";
    cout << "Parallel Pop time: " << pop_time.count() << " seconds\n";
    cout << "Note: Atomic operations ensure safe concurrent access (Lecture #5)\n";

    return 0;
}

Overwriting part1_parallel_stack.cu


In [5]:
!nvcc -std=c++17 part1_parallel_stack.cu -o part1_parallel_stack
!./part1_parallel_stack


Parallel Stack (LIFO) on GPU
Parallel Push time: 0.055839 seconds
Parallel Pop time: 5.384e-06 seconds
Note: Atomic operations ensure safe concurrent access (Lecture #5)


In [9]:
%%writefile part2_parallel_queue.cu
// part2_parallel_queue.cu — Практическая работа №5: Параллельная очередь (FIFO) на CUDA
#include <iostream>      // Для вывода результатов и отладки (Лекция №1: базовый ввод-вывод)
#include <random>        // Для генерации случайных чисел: mt19937 и uniform_int_distribution (Лекция №1: важность случайных данных для тестов)
#include <chrono>        // Для замера времени выполнения (Лекция №4: анализ производительности)
#include <cuda_runtime.h> // CUDA API: память, ядра, атомарные операции (Лекция №3: основы CUDA, Лекция №5: параллельные структуры)

using namespace std;     // Упрощает код (Лекция №1: стандартная практика)

#define CUDA_CHECK(err) do { \
    cudaError_t local_err = (err); \
    if (local_err != cudaSuccess) { \
        cerr << "CUDA error: " << cudaGetErrorString(local_err) << " at line " << __LINE__ << endl; \
        exit(1); \
    } \
} while(0)

// Константы
const int MAX_QUEUE_SIZE = 1000000;  // Максимальный размер очереди (Лекция №5: фиксированный размер для GPU)
const int BLOCK_SIZE = 256;          // Размер блока (Лекция №3: оптимальный размер для occupancy)

// Глобальная память для очереди и указателей (Лекция №5: глобальная память для общей структуры)
__device__ int queue[MAX_QUEUE_SIZE];
__device__ int head = 0;  // Указатель начала (для dequeue)
__device__ int tail = 0;  // Указатель конца (для enqueue)

// Ядро для enqueue (добавление в конец) — Лекция №5: безопасное добавление элементов
__global__ void parallel_enqueue(int *values, int num_enqueue) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;  // Глобальный индекс потока (Лекция №3: индексация)
    if (idx >= num_enqueue) return;  // Проверка границ (Лекция №3: безопасность потоков)

    int val = values[idx];  // Значение для enqueue (Лекция №5: данные для очереди)

    // Атомарное увеличение tail и запись (Лекция №5: atomicAdd для безопасного enqueue)
    int pos = atomicAdd(&tail, 1);  // Атомарно увеличиваем tail
    if (pos < MAX_QUEUE_SIZE) {     // Проверка переполнения (Лекция №5: защита от переполнения)
        queue[pos] = val;           // Запись значения в очередь (Лекция №4: глобальная память)
    }
}

// Ядро для dequeue (извлечение из начала) — Лекция №5: безопасное извлечение элементов
__global__ void parallel_dequeue(int *results, int num_dequeue) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;  // Глобальный индекс
    if (idx >= num_dequeue) return;  // Проверка границ

    int pos = atomicAdd(&head, 1);  // Атомарно увеличиваем head (Лекция №5: atomicAdd для dequeue)
    if (pos < tail) {               // Проверка на наличие элементов (Лекция №5: защита от underflow)
        results[idx] = queue[pos];  // Извлечение значения (Лекция №5: атомарный dequeue)
    } else {
        results[idx] = -1;          // Ошибка — очередь пуста (Лекция №5: обработка underflow)
    }
}

int main() {
    cout << "Parallel Queue (FIFO) on GPU\n";  // Заголовок (Лекция №5: демонстрация параллельной очереди)

    const int NUM_OPERATIONS = 1000000;  // Количество операций enqueue/dequeue (Лекция №5: большое количество для теста)
    int *h_values, *d_values;            // Массивы для значений
    int *h_results, *d_results;          // Результаты dequeue

    // Выделение на CPU
    h_values = new int[NUM_OPERATIONS];
    h_results = new int[NUM_OPERATIONS];

    // Заполнение случайными значениями (Лекция №1: случайные данные для тестов)
    mt19937 gen(time(nullptr));
    uniform_int_distribution<int> dist(1, 1000000);
    for (int i = 0; i < NUM_OPERATIONS; ++i) h_values[i] = dist(gen);

    // Выделение на GPU
    CUDA_CHECK(cudaMalloc(&d_values, NUM_OPERATIONS * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_results, NUM_OPERATIONS * sizeof(int)));

    // Копирование значений на GPU
    CUDA_CHECK(cudaMemcpy(d_values, h_values, NUM_OPERATIONS * sizeof(int), cudaMemcpyHostToDevice));

    dim3 threads(BLOCK_SIZE);
    dim3 blocks((NUM_OPERATIONS + BLOCK_SIZE - 1) / BLOCK_SIZE);

    // Замер времени enqueue
    auto start_enqueue = chrono::high_resolution_clock::now();
    parallel_enqueue<<<blocks, threads>>>(d_values, NUM_OPERATIONS);
    CUDA_CHECK(cudaDeviceSynchronize());
    auto end_enqueue = chrono::high_resolution_clock::now();
    chrono::duration<double> enqueue_time = end_enqueue - start_enqueue;

    // Замер времени dequeue
    auto start_dequeue = chrono::high_resolution_clock::now();
    parallel_dequeue<<<blocks, threads>>>(d_results, NUM_OPERATIONS);
    CUDA_CHECK(cudaDeviceSynchronize());
    auto end_dequeue = chrono::high_resolution_clock::now();
    chrono::duration<double> dequeue_time = end_dequeue - start_dequeue;

    // Копирование результатов dequeue обратно
    CUDA_CHECK(cudaMemcpy(h_results, d_results, NUM_OPERATIONS * sizeof(int), cudaMemcpyDeviceToHost));

    // Освобождение памяти
    delete[] h_values;
    delete[] h_results;
    CUDA_CHECK(cudaFree(d_values));
    CUDA_CHECK(cudaFree(d_results));

    cout << "Parallel Enqueue time: " << enqueue_time.count() << " seconds\n";
    cout << "Parallel Dequeue time: " << dequeue_time.count() << " seconds\n";
    cout << "Note: Atomic operations ensure safe concurrent access (Lecture #5)\n";

    return 0;
}

Overwriting part2_parallel_queue.cu


In [10]:
!nvcc -std=c++17 part2_parallel_queue.cu -o part2_parallel_queue
!./part2_parallel_queue


Parallel Queue (FIFO) on GPU
Parallel Enqueue time: 0.00842917 seconds
Parallel Dequeue time: 3.502e-06 seconds
Note: Atomic operations ensure safe concurrent access (Lecture #5)
